## 1. Imports & Configuration

In [ ]:
import os, json, pickle, time, warnings, re, requests
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch, torch.nn as nn
from transformers import BertTokenizer, BertModel

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120, 'figure.facecolor': 'white'})

PROJECT_ROOT    = Path(os.getcwd()).parent
DATA_DIR        = PROJECT_ROOT / 'data'
RESULTS_DIR     = PROJECT_ROOT / 'results'
MODELS_DIR      = PROJECT_ROOT / 'models'

STARPEP_CSV     = DATA_DIR / 'StarPep_APD_only.csv'
PREDICTIONS_CSV = RESULTS_DIR / 'test_predictions.csv'
EMBEDDINGS_NPY  = DATA_DIR / 'starpep_embeddings_cosine.npy'
METADATA_PKL    = DATA_DIR / 'starpep_metadata.pkl'
OUTPUT_CSV      = RESULTS_DIR / 'amp_rag_results.csv'

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

os.environ['PYTORCH_MPS_HIGH_WATERMARK_RATIO'] = '0.0'

PROTBERT_ID = 'Rostlab/prot_bert_bfd'
MAX_LEN     = 128
EMBED_BATCH = 16
TOP_K       = 3

OLLAMA_BASE_URL = 'http://localhost:11434'
OLLAMA_MODEL    = 'mistral:7b-instruct'
LLM_TEMPERATURE = 0
LLM_MAX_TOKENS  = 800

if torch.cuda.is_available():           DEVICE = 'cuda'
elif torch.backends.mps.is_available(): DEVICE = 'mps'
else:                                   DEVICE = 'cpu'

def clear_memory():
    if DEVICE == 'mps':  torch.mps.empty_cache()
    elif DEVICE == 'cuda': torch.cuda.empty_cache()

print(f'Device     : {DEVICE}')
print(f'Output CSV : {OUTPUT_CSV.name}')
print(f'LLM        : {OLLAMA_MODEL}  @ {OLLAMA_BASE_URL}')
print(f'Temperature: {LLM_TEMPERATURE}  (deterministic)')
print(f'Similarity : cosine (L2-normalised dot product, magnitude-invariant)')

/Users/deepugeorgejacob/Projects/amp-rag/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Device     : mps
Output CSV : amp_rag_results.csv
LLM        : mistral:7b-instruct  @ http://localhost:11434
Temperature: 0  (deterministic)
Similarity : cosine (L2-normalised dot product, magnitude-invariant)


## 1b. Ollama Health Check

Verifies that Ollama is running locally and `mistral:7b-instruct` is available.  
Start Ollama with: `ollama serve` (or open the Ollama desktop app).

In [ ]:
def ollama_health_check(base_url: str, model: str) -> bool:
    """Return True if Ollama is running and the model is available."""
    try:
        # Check server is up
        r = requests.get(f"{base_url}/api/tags", timeout=5)
        r.raise_for_status()
        models_available = [m["name"] for m in r.json().get("models", [])]
        print(f"Ollama server   : ✅ running")
        print(f"Models loaded   : {models_available}")

        if model in models_available:
            print(f"Target model    : ✅ {model} ready")
            return True
        else:
            print(f"Target model    : ❌ '{model}' not found")
            print(f"  Run: ollama pull {model}")
            return False

    except requests.exceptions.ConnectionError:
        print("❌ Ollama not reachable — start it with: ollama serve")
        return False
    except Exception as e:
        print(f"❌ Health check failed: {e}")
        return False

OLLAMA_READY = ollama_health_check(OLLAMA_BASE_URL, OLLAMA_MODEL)

if OLLAMA_READY:
    # Quick warm-up call to load model weights into memory
    print("\nWarm-up call (loads model weights)...")
    r = requests.post(
        f"{OLLAMA_BASE_URL}/api/chat",
        json={
            "model":    OLLAMA_MODEL,
            "messages": [{"role": "user", "content": "What is an antimicrobial peptide? One sentence."}],
            "stream":   False,
            "options":  {"temperature": LLM_TEMPERATURE, "num_predict": 60},
        },
        timeout=120,
    )
    warmup_reply = r.json()["message"]["content"].strip()
    print(f"✅ Model warm-up OK: {warmup_reply[:120]}")
else:
    print("\n⚠️  Fix Ollama before running Section 6.")

Ollama server   : ✅ running
Models loaded   : ['mistral:7b-instruct']
Target model    : ✅ mistral:7b-instruct ready

Warm-up call (loads model weights)...
✅ Model warm-up OK: Antimicrobial peptides are small, naturally occurring proteins that possess the ability to destroy microorganisms such a


In [ ]:
starpep_df = pd.read_csv(STARPEP_CSV)

starpep_df["sequence"] = starpep_df["sequence"].astype(str).str.strip().str.upper()
starpep_df = starpep_df[starpep_df["sequence"].str.len() > 0].reset_index(drop=True)

meta_cols = ["assessedAgainst", "relatedTo", "producedBy", "modifiedBy",
             "compiledIn", "linkedTo", "constitutedBy"]
for col in meta_cols:
    if col in starpep_df.columns:
        starpep_df[col] = starpep_df[col].fillna("").astype(str)

print(f"StarPep records     : {len(starpep_df):,}")
print(f"Columns             : {starpep_df.columns.tolist()}")
print(f"\nSequence length stats:")
print(starpep_df["length"].describe().round(1))
starpep_df.head(3)

StarPep records     : 2,949
Columns             : ['id', 'sequence', 'length', 'assessedAgainst', 'compiledIn', 'constitutedBy', 'linkedTo', 'modifiedBy', 'producedBy', 'relatedTo']

Sequence length stats:
count    2949.0
mean       30.8
std        16.4
min         2.0
25%        20.0
50%        28.0
75%        38.0
max       100.0
Name: length, dtype: float64


,id,sequence,length,assessedAgainst,compiledIn,constitutedBy,linkedTo,modifiedBy,producedBy,relatedTo
0,starPep_00000,GIGAVLKVLTTGLPALISWIKRKRQQ,26,HIV|Xanthomonas oryzae|Propionibacterium acnes...,HIPdb|ADAM|DRAMP_Patent|StraPep|AntiTbPdb|RAPD...,P at position 14 is D Proline|V at position 8 ...,PubMed: 23150226|PubMed: 10217411|PubMed: 2389...,Amidation|Lac=P-Amidophenyl-Β-D-Lactopyranosid...,Insecta|Apis cerana|Polistes|Apis mellifera|Sy...,Antifungal|Antibacterial|Antibiofilm|Toxic|Ant...
1,starPep_00001,GIGKFLHSAKKFGKAFVGEIMNS,23,Leukemia cancer|Serratia marcescens|Enterococc...,AMSDb|SATPdb|ADAM|AntiTbPdb|LAMP_Experimental|...,E at position 19 is D Glutamic Acid|K at posit...,PubMed: 9090128|DOI: 10.1093/jac/dkl120|PubMed...,Amidation,Synthetic construct|Xenopus ruwenzoriensis|Amp...,Toxic to mammals|Antiparasitic|Antibacterial|A...
2,starPep_00002,ILPWKWPWWPWRR,13,Streptococcus pyogenes|Sheep erythrocytes|Micr...,HIPdb|StraPep|AMSDb|CAMP_Validated|CAMP_Patent...,I at position 1 is D Isoleucine|W at position ...,PubMed: 11061996|PubMed: 16998183|PubMed: 1092...,Amidation|Acetylation,Bos taurus|Streptococcus equi|Synthetic construct,Surface-immobilization|Antibiofilm|Antileishma...


In [ ]:
tokenizer = BertTokenizer.from_pretrained(
    PROTBERT_ID, do_lower_case=False, cache_dir=str(MODELS_DIR)
)
encoder = BertModel.from_pretrained(PROTBERT_ID, cache_dir=str(MODELS_DIR))
encoder = encoder.to(DEVICE)
encoder.eval()

total_params = sum(p.numel() for p in encoder.parameters())
print(f"✅ ProtBERT-BFD loaded  ({total_params/1e6:.0f}M params, device={DEVICE})")

✅ ProtBERT-BFD loaded  (420M params, device=mps)


In [ ]:
@torch.no_grad()
def embed_sequences(sequences: list[str],
                    tokenizer,
                    model,
                    device: str,
                    batch_size: int = EMBED_BATCH,
                    max_len: int = MAX_LEN) -> np.ndarray:
    """
    Embed a list of amino-acid sequences using ProtBERT-BFD.
    Returns a (N, 1024) float32 numpy array of [CLS] embeddings.
    """
    all_embeddings = []
    n_batches = (len(sequences) + batch_size - 1) // batch_size

    for i in range(0, len(sequences), batch_size):
        batch = sequences[i : i + batch_size]
        batch_num = i // batch_size + 1

        spaced = [" ".join(list(seq.upper().replace(" ", ""))) for seq in batch]

        enc = tokenizer(
            spaced,
            max_length=max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        input_ids = enc["input_ids"].to(device)
        attn_mask = enc["attention_mask"].to(device)

        out = model(input_ids=input_ids, attention_mask=attn_mask)
        cls_emb = out.last_hidden_state[:, 0, :].cpu().float().numpy()
        all_embeddings.append(cls_emb)

        del input_ids, attn_mask, out
        clear_memory()

        if batch_num % 20 == 0 or batch_num == n_batches:
            pct = batch_num / n_batches * 100
            print(f"  [{batch_num:4d}/{n_batches}]  {pct:5.1f}%  "
                  f"processed {min(i+batch_size, len(sequences)):4d}/{len(sequences)} seqs",
                  flush=True)

    return np.vstack(all_embeddings)   # (N, 1024)


print("Embedding function defined.")

Embedding function defined.


In [ ]:
if EMBEDDINGS_NPY.exists() and METADATA_PKL.exists():
    print('Cached embeddings found — loading...')
    store_embeddings = np.load(EMBEDDINGS_NPY)
    with open(METADATA_PKL, 'rb') as f:
        store_metadata = pickle.load(f)
    print(f'   Shape : {store_embeddings.shape}  (L2-normalised unit vectors)')
    print(f'   Meta  : {len(store_metadata)} records')
else:
    print(f'Embedding {len(starpep_df):,} StarPep sequences...')
    t0 = time.time()
    raw_embeddings = embed_sequences(starpep_df['sequence'].tolist(),
                                     tokenizer, encoder, DEVICE)

    norms = np.linalg.norm(raw_embeddings, axis=1, keepdims=True)
    store_embeddings = raw_embeddings / np.clip(norms, 1e-9, None)
    np.save(EMBEDDINGS_NPY, store_embeddings)
    print(f'\nL2-normalised embeddings saved -> {EMBEDDINGS_NPY.name}  {store_embeddings.shape}')

    store_metadata = []
    for _, row in starpep_df.iterrows():
        store_metadata.append({
            'id':              row['id'],
            'sequence':        row['sequence'],
            'length':          row.get('length', len(row['sequence'])),
            'assessedAgainst': row.get('assessedAgainst', ''),
            'relatedTo':       row.get('relatedTo', ''),
            'producedBy':      row.get('producedBy', ''),
            'modifiedBy':      row.get('modifiedBy', ''),
            'compiledIn':      row.get('compiledIn', ''),
            'linkedTo':        row.get('linkedTo', ''),
        })
    with open(METADATA_PKL, 'wb') as f:
        pickle.dump(store_metadata, f)
    print(f'Metadata saved -> {METADATA_PKL.name}  ({len(store_metadata)} records)')
    print(f'Time : {(time.time()-t0)/60:.1f} min')

sample_norms = np.linalg.norm(store_embeddings[:5], axis=1)
print(f'\nSample L2 norms (should be 1.0): {sample_norms.round(6)}')
print(f'Vector store ready — {store_embeddings.shape[0]:,} unit-norm AMPs @ {store_embeddings.shape[1]}-d')
print('Similarity: cosine  (store @ query.T  since both are unit vectors)')

Cached embeddings found — loading...
   Shape : (2949, 1024)  (L2-normalised unit vectors)
   Meta  : 2949 records

Sample L2 norms (should be 1.0): [1. 1. 1. 1. 1.]
Vector store ready — 2,949 unit-norm AMPs @ 1024-d
Similarity: cosine  (store @ query.T  since both are unit vectors)


In [ ]:
pred_df = pd.read_csv(PREDICTIONS_CSV)
print(f"Total test predictions : {len(pred_df):,}")
print(f"Class distribution     : {pred_df['predicted'].value_counts().to_dict()}")

pos_df = pred_df[pred_df["predicted"] == 1].reset_index(drop=True)
print(f"\nPositive AMP predictions : {len(pos_df):,}")
print(f"  True Positives (TP)    : {(pos_df['result']=='TP').sum()}")
print(f"  False Positives (FP)   : {(pos_df['result']=='FP').sum()}")
pos_df.head(5)

Total test predictions : 712
Class distribution     : {1: 358, 0: 354}

Positive AMP predictions : 358
  True Positives (TP)    : 323
  False Positives (FP)   : 35


,sequence,aa_length,label,prob_amp,predicted,correct,result
0,GIVDFAKKVVGGIRNALGI,19,1,0.799875,1,1,TP
1,FISAIASMLGKFL,13,1,0.799397,1,1,TP
2,FLPLVGKILSGLI,13,1,0.798918,1,1,TP
3,GLLSALRKMIPHILSHIKK,19,1,0.800874,1,1,TP
4,AALKGCWTKSIPPKPCSGKR,20,1,0.799576,1,1,TP


In [ ]:
def retrieve_top_k(query_seq, store_embs, store_meta, tokenizer, model, device, top_k=TOP_K):
    query_emb = embed_sequences([query_seq], tokenizer, model, device, batch_size=1)
    query_unit = query_emb / np.clip(np.linalg.norm(query_emb), 1e-9, None)

    cosine_scores = (store_embs @ query_unit.T).squeeze()
    top_idx = np.argsort(cosine_scores)[::-1][:top_k]
    results = []
    for rank, idx in enumerate(top_idx, start=1):
        meta = dict(store_meta[idx])
        meta['rank']        = rank
        meta['cosine_sim']  = round(float(cosine_scores[idx]), 6)
        meta['is_self_match'] = (meta['cosine_sim'] >= 0.9999 and
                                  store_meta[idx]['sequence'].upper() == query_seq.upper())
        results.append(meta)
    return results

print('Retrieval function ready — cosine similarity, self-match included (score=1.0 flags known AMP).')

Retrieval function ready — cosine similarity, self-match included (score=1.0 flags known AMP).


In [ ]:
demo_seq = pos_df['sequence'].iloc[0]
print(f"Query : {demo_seq}  (len={len(demo_seq)} aa,  prob={pos_df['prob_amp'].iloc[0]:.4f})")
print('\nRetrieving top-3 via cosine similarity (self-match shown if present)...')
demo_results = retrieve_top_k(demo_seq, store_embeddings, store_metadata,
                               tokenizer, encoder, DEVICE)
print(f"\n{'Rank':<5} {'CosineSim':>10}  {'Self?':>5}  {'ID':<18} {'Seq':<30} {'Activities (first 3)'}")
print('-' * 115)
for r in demo_results:
    acts = ', '.join(r['relatedTo'].split('|')[:3]) if r['relatedTo'] else 'N/A'
    flag = ' ✓ SELF' if r['is_self_match'] else ''
    print(f"#{r['rank']:<4} {r['cosine_sim']:>10.6f}  {'YES' if r['is_self_match'] else 'no':>5}  {r['id']:<18} {r['sequence'][:28]:<30} {acts}{flag}")
print()
self_match = next((r for r in demo_results if r['is_self_match']), None)
if self_match:
    print(f"✅ Exact self-match found at rank {self_match['rank']} — this sequence IS in StarPep APD (cosine=1.0)")
else:
    print('ℹ️  No exact self-match in top-3 — sequence is not in StarPep APD (or only partially similar)')

Query : GIVDFAKKVVGGIRNALGI  (len=19 aa,  prob=0.7999)

Retrieving top-3 via cosine similarity (self-match shown if present)...
  [   1/1]  100.0%  processed    1/1 seqs

Rank   CosineSim  Self?  ID                 Seq                            Activities (first 3)
-------------------------------------------------------------------------------------------------------------------
#1      1.000000    YES  starPep_00309      GIVDFAKKVVGGIRNALGI            Anti-Gram negative, Anti-Gram positive, Antimicrobial ✓ SELF
#2      0.990304     no  starPep_00302      GILDFAKTVVGGIRNALGI            Antimicrobial, Anti-Gram positive, Anti-Gram negative
#3      0.979714     no  starPep_00295      GFVDLAKKVVGGIRNALGI            Antibacterial, Antimicrobial, Anti-Gram negative

✅ Exact self-match found at rank 1 — this sequence IS in StarPep APD (cosine=1.0)


In [ ]:
print(f"Retrieving top-{TOP_K} similar AMPs for {len(pos_df):,} positive predictions...")
print("(Each query embeds one sequence — may take a few minutes)\n")

retrieval_results = {}

for idx, row in pos_df.iterrows():
    seq = row["sequence"]
    hits = retrieve_top_k(seq, store_embeddings, store_metadata,
                          tokenizer, encoder, DEVICE)
    retrieval_results[seq] = {
        "sequence":  seq,
        "prob_amp":  row["prob_amp"],
        "true_label":int(row["label"]),
        "result":    row["result"],
        "hits":      hits,
    }
    if (idx + 1) % 50 == 0 or (idx + 1) == len(pos_df):
        print(f"  [{idx+1:>3}/{len(pos_df)}] done", flush=True)

clear_memory()
print(f"\n✅ Retrieval complete for {len(retrieval_results):,} sequences.")

Retrieving top-3 similar AMPs for 358 positive predictions...
(Each query embeds one sequence — may take a few minutes)

  [   1/1]  100.0%  processed    1/1 seqs
  [   1/1]  100.0%  processed    1/1 seqs
  [   1/1]  100.0%  processed    1/1 seqs
  [   1/1]  100.0%  processed    1/1 seqs
  [   1/1]  100.0%  processed    1/1 seqs
  [   1/1]  100.0%  processed    1/1 seqs
  [   1/1]  100.0%  processed    1/1 seqs
  [   1/1]  100.0%  processed    1/1 seqs
  [   1/1]  100.0%  processed    1/1 seqs
  [   1/1]  100.0%  processed    1/1 seqs
  [   1/1]  100.0%  processed    1/1 seqs
  [   1/1]  100.0%  processed    1/1 seqs
  [   1/1]  100.0%  processed    1/1 seqs
  [   1/1]  100.0%  processed    1/1 seqs
  [   1/1]  100.0%  processed    1/1 seqs
  [   1/1]  100.0%  processed    1/1 seqs
  [   1/1]  100.0%  processed    1/1 seqs
  [   1/1]  100.0%  processed    1/1 seqs
  [   1/1]  100.0%  processed    1/1 seqs
  [   1/1]  100.0%  processed    1/1 seqs
  [   1/1]  100.0%  processed    1/1 se

In [ ]:
HYDROPHOBIC = set("AVILMFYW")
CHARGED_POS = set("KRH")
CHARGED_NEG = set("DE")

def compute_properties(seq: str) -> dict:
    seq = seq.upper()
    n = len(seq)
    hydrophobic_count = sum(1 for aa in seq if aa in HYDROPHOBIC)
    pos_charge  = sum(1 for aa in seq if aa in CHARGED_POS)
    neg_charge  = sum(1 for aa in seq if aa in CHARGED_NEG)
    net_charge  = pos_charge - neg_charge
    return {
        "length":          n,
        "hydrophobicity":  round(hydrophobic_count / n * 100, 1),
        "net_charge":      net_charge,
        "pos_charged_pct": round(pos_charge / n * 100, 1),
        "neg_charged_pct": round(neg_charge / n * 100, 1),
    }


def format_activities(pipe_str: str, max_items: int = 6) -> str:
    items = [x.strip() for x in pipe_str.split("|") if x.strip()][:max_items]
    return ", ".join(items) if items else "Not specified"


def format_targets(pipe_str: str, max_items: int = 8) -> str:
    items = [x.strip() for x in pipe_str.split("|") if x.strip()][:max_items]
    return ", ".join(items) if items else "Not specified"


def build_prompt(query_seq: str,
                 prob_amp: float,
                 hits: list[dict]) -> str:
    """
    Construct an expert level biological prompt for the LLM.
    """
    props = compute_properties(query_seq)

    # Format the three retrieved AMPs
    hit_sections = []
    for h in hits:
        activities = format_activities(h.get("relatedTo", ""))
        targets    = format_targets(h.get("assessedAgainst", ""))
        source     = format_activities(h.get("producedBy", ""), max_items=3)
        mods       = format_activities(h.get("modifiedBy", ""), max_items=3)

        hit_sections.append(
            f"  [{h['rank']}] ID: {h['id']}  Similarity: {h['cosine_sim']:.4f}\n"
            f"      Sequence     : {h['sequence']}  (length: {h.get('length', len(h['sequence']))} aa)\n"
            f"      Activities   : {activities}\n"
            f"      Target organisms: {targets}\n"
            f"      Biological source: {source}\n"
            f"      Modifications : {mods if mods != 'Not specified' else 'None reported'}"
        )

    hits_text = "\n\n".join(hit_sections)

    prompt = f"""You are an expert computational biologist specialising in antimicrobial peptides (AMPs) and protein bioinformatics.

A machine-learning classifier (ProtBERT-BFD fine-tuned on the Veltri dataset) has predicted the following peptide sequence to be an antimicrobial peptide with probability {prob_amp:.4f}:

QUERY SEQUENCE: {query_seq}

PHYSICOCHEMICAL PROPERTIES:
  • Length         : {props['length']} amino acids
  • Hydrophobicity : {props['hydrophobicity']}% hydrophobic residues ({', '.join(HYDROPHOBIC & set(query_seq.upper()))})
  • Net charge     : {props['net_charge']:+d} (Positive residues: {props['pos_charged_pct']}%, Negative: {props['neg_charged_pct']}%)

TOP-3 MOST SIMILAR KNOWN AMPs (cosine similarity in ProtBERT-BFD embedding space):

{hits_text}

Based on the query sequence and its closest known AMP neighbours, provide a comprehensive biological explanation covering:

1. ANTIMICROBIAL MECHANISM — What membrane-disrupting or intracellular mechanisms are most likely? Relate to physicochemical properties (charge, hydrophobicity, amphipathicity).
2. PREDICTED TARGET SPECTRUM — Which organisms or pathogens is this peptide most likely active against, based on the neighbours' target profiles?
3. STRUCTURAL FEATURES — What structural motifs (alpha-helix, beta-sheet, random coil) and key residues likely drive activity?
4. BIOLOGICAL ORIGIN & EVOLUTION — What biological context might this peptide arise from, based on the source organisms of similar peptides?
5. THERAPEUTIC POTENTIAL — What biomedical applications (antibacterial, antifungal, antiviral, anticancer) are most supported by the evidence?
6. RESEARCH RECOMMENDATIONS — What experimental assays would you recommend to validate this peptide's activity?

Be specific, cite amino acid positions where relevant, and ground your explanation in the retrieved AMP data."""

    return prompt


print("Prompt builder defined.")
print("\nSample prompt for demo sequence:")
print("-"*60)
sample_prompt = build_prompt(
    demo_seq, pos_df['prob_amp'].iloc[0], demo_results
)
print(sample_prompt[:600], "...")

Prompt builder defined.

Sample prompt for demo sequence:
------------------------------------------------------------
You are an expert computational biologist specialising in antimicrobial peptides (AMPs) and protein bioinformatics.

A machine-learning classifier (ProtBERT-BFD fine-tuned on the Veltri dataset) has predicted the following peptide sequence to be an antimicrobial peptide with probability 0.7999:

QUERY SEQUENCE: GIVDFAKKVVGGIRNALGI

PHYSICOCHEMICAL PROPERTIES:
  • Length         : 19 amino acids
  • Hydrophobicity : 52.6% hydrophobic residues (A, V, F, L, I)
  • Net charge     : +2 (Positive residues: 15.8%, Negative: 5.3%)

TOP-3 MOST SIMILAR KNOWN AMPs (cosine similarity in ProtBERT-BFD embed ...


In [ ]:
print(sample_prompt)

You are an expert computational biologist specialising in antimicrobial peptides (AMPs) and protein bioinformatics.

A machine-learning classifier (ProtBERT-BFD fine-tuned on the Veltri dataset) has predicted the following peptide sequence to be an antimicrobial peptide with probability 0.7999:

QUERY SEQUENCE: GIVDFAKKVVGGIRNALGI

PHYSICOCHEMICAL PROPERTIES:
  • Length         : 19 amino acids
  • Hydrophobicity : 52.6% hydrophobic residues (A, V, F, L, I)
  • Net charge     : +2 (Positive residues: 15.8%, Negative: 5.3%)

TOP-3 MOST SIMILAR KNOWN AMPs (cosine similarity in ProtBERT-BFD embedding space):

  [1] ID: starPep_00309  Similarity: 1.0000
      Sequence     : GIVDFAKKVVGGIRNALGI  (length: 19 aa)
      Activities   : Anti-Gram negative, Anti-Gram positive, Antimicrobial, Antibacterial
      Target organisms: Streptococcus uberis, Leuconostoc mesenteroides, Micrococcus luteus
      Biological source: Uperoleia inundata
      Modifications : Amidation

  [2] ID: starPep_00302  

In [ ]:
# ── Ollama LLM inference (local, temperature=0 → deterministic) ────────────────
def query_llm(prompt: str,
              model:  str    = OLLAMA_MODEL,
              base_url: str  = OLLAMA_BASE_URL,
              max_tokens: int = LLM_MAX_TOKENS,
              temperature: float = LLM_TEMPERATURE,
              retries: int = 3) -> str:
    """
    Send a chat prompt to the local Ollama instance.
    temperature=0 → greedy decoding → fully deterministic output.
    Returns the generated explanation text.
    """
    payload = {
        "model":  model,
        "stream": False,
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are an expert computational biologist specialising in "
                    "antimicrobial peptides (AMPs). Provide detailed, scientifically "
                    "accurate biological explanations using structured headings. "
                    "Base your answer strictly on the provided sequence data and "
                    "retrieved similar AMPs."
                ),
            },
            {"role": "user", "content": prompt},
        ],
        "options": {
            "temperature": temperature,
            "num_predict": max_tokens,
            "top_p":       1.0,
            "top_k":       1,
        },
    }

    for attempt in range(1, retries + 1):
        try:
            resp = requests.post(
                f"{base_url}/api/chat",
                json=payload,
                timeout=300,
            )
            resp.raise_for_status()
            return resp.json()["message"]["content"].strip()
        except requests.exceptions.Timeout:
            print(f"  ⚠️  Timeout on attempt {attempt}/{retries}. Retrying...")
        except requests.exceptions.ConnectionError:
            print(f"  ❌ Ollama not reachable. Is 'ollama serve' running?")
            return "[LLM ERROR] Ollama not reachable."
        except Exception as e:
            wait = 2 ** attempt
            print(f"  ⚠️  Attempt {attempt}/{retries} failed: {e}. Retrying in {wait}s...")
            time.sleep(wait)

    return f"[LLM ERROR] Could not generate explanation after {retries} attempts."


print(f"✅ Ollama LLM function ready")
print(f"   Model       : {OLLAMA_MODEL}")
print(f"   Temperature : {LLM_TEMPERATURE}  (greedy / deterministic)")
print(f"   top_k       : 1  (greedy decoding)")
print(f"   Max tokens  : {LLM_MAX_TOKENS}")

✅ Ollama LLM function ready
   Model       : mistral:7b-instruct
   Temperature : 0  (greedy / deterministic)
   top_k       : 1  (greedy decoding)
   Max tokens  : 800


### Generate explanations for all positive AMP predictions

In [ ]:
if not OLLAMA_READY:
    raise RuntimeError("Ollama is not running. Start it with: ollama serve")

print(f"Generating biological explanations for {len(retrieval_results):,} sequences...")
print(f"LLM   : {OLLAMA_MODEL}  (local Ollama)")
print(f"Temp  : {LLM_TEMPERATURE}  — deterministic greedy decoding\n")

explanations = {}
failed_seqs  = []
t_start = time.time()

for i, (seq, data) in enumerate(list(retrieval_results.items()), start=1):
    t0          = time.time()
    prompt      = build_prompt(seq, data["prob_amp"], data["hits"])
    explanation = query_llm(prompt)
    elapsed_seq = time.time() - t0

    explanations[seq] = explanation

    status = "✅" if "[LLM ERROR]" not in explanation else "❌"
    if "[LLM ERROR]" in explanation:
        failed_seqs.append(seq)

    print(f"  {status} [{i:>3}/{len(retrieval_results)}]  "
          f"{elapsed_seq:5.1f}s  seq={seq[:18]:<18}  "
          f"words={len(explanation.split()):>4}", flush=True)

total_elapsed = time.time() - t_start
print(f"\n{'─'*60}")
print(f"✅ Explanations generated : {len(explanations) - len(failed_seqs):,}")
print(f"❌ Failed                : {len(failed_seqs)}")
print(f"⏱  Total time            : {total_elapsed/60:.1f} min  "
      f"({total_elapsed/len(retrieval_results):.1f}s per sequence)")

Generating biological explanations for 358 sequences...
LLM   : mistral:7b-instruct  (local Ollama)
Temp  : 0  — deterministic greedy decoding

  ✅ [  1/358]   40.1s  seq=GIVDFAKKVVGGIRNALG  words= 413
  ✅ [  2/358]   33.0s  seq=FISAIASMLGKFL       words= 347
  ✅ [  3/358]   40.8s  seq=FLPLVGKILSGLI       words= 423
  ✅ [  4/358]   36.3s  seq=GLLSALRKMIPHILSHIK  words= 354
  ✅ [  5/358]   34.9s  seq=AALKGCWTKSIPPKPCSG  words= 365
  ✅ [  6/358]   41.5s  seq=YVPLPNVPQPGRRPFPTF  words= 418
  ✅ [  7/358]   41.5s  seq=FLPILAGLAAKLVPKVFC  words= 415
  ✅ [  8/358]   40.4s  seq=FLGGILNTITGLL       words= 457
  ✅ [  9/358]   41.5s  seq=GPVSAVLTELRCTCLRVT  words= 345
  ✅ [ 10/358]   41.2s  seq=FFRHLFRGAKAIFRGARQ  words= 409
  ✅ [ 11/358]   40.8s  seq=QNGGNQQGFGGEGIETFL  words= 445
  ✅ [ 12/358]   41.4s  seq=RSALSCQMCELVVKKYEG  words= 435
  ✅ [ 13/358]   41.0s  seq=ITSVSWCTPGCTSEGGGS  words= 450
  ✅ [ 14/358]   41.8s  seq=ELTEAQRRGLQVALEEFH  words= 421
  ✅ [ 15/358]   40.4s  seq=PPCRGIFCRRVGSSSAI

In [ ]:
def fmt(pipe_str, max_items=8):
    items = [x.strip() for x in str(pipe_str).split('|') if x.strip()][:max_items]
    return ', '.join(items)

rows = []
for idx, (seq, data) in enumerate(retrieval_results.items()):
    explanation = explanations.get(seq, '[No explanation generated]')
    props = compute_properties(seq)
    row = {
        'seq_idx':           idx + 1,
        'query_sequence':    seq,
        'query_length':      props['length'],
        'prob_amp':          round(data['prob_amp'], 6),
        'true_label':        data['true_label'],
        'prediction_result': data['result'],
        'in_starpep':        any(h.get('is_self_match', False) for h in data['hits']),
        'hydrophobicity_pct':props['hydrophobicity'],
        'net_charge':        props['net_charge'],
        'pos_charge_pct':    props['pos_charged_pct'],
        'neg_charge_pct':    props['neg_charged_pct'],
    }
    for n in range(1, TOP_K + 1):
        p = f'hit{n}_'
        if n <= len(data['hits']):
            h = data['hits'][n - 1]
            row[p+'starpep_id']    = h['id']
            row[p+'is_self_match'] = h.get('is_self_match', False)
            row[p+'cosine_sim']   = round(h['cosine_sim'], 4)
            row[p+'sequence']      = h['sequence']
            row[p+'length']        = h.get('length', len(h['sequence']))
            row[p+'activities']    = fmt(h.get('relatedTo',       ''))
            row[p+'targets']       = fmt(h.get('assessedAgainst', ''), max_items=10)
            row[p+'source']        = fmt(h.get('producedBy',      ''), max_items=4)
            row[p+'modifications'] = fmt(h.get('modifiedBy',      ''), max_items=4)
        else:
            for col in ['starpep_id','cosine_sim','sequence','length',
                        'activities','targets','source','modifications']:
                row[p+col] = ''
    row['biological_explanation'] = explanation
    rows.append(row)

results_df = pd.DataFrame(rows)
results_df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')

print(f'Results saved -> {OUTPUT_CSV}')
print(f'   Rows    : {len(results_df)}')
print(f'   Columns : {len(results_df.columns)}')
print(f'   Size    : {OUTPUT_CSV.stat().st_size / 1024:.1f} KB')
print()
print('Column layout:')
print('  [query]           seq_idx  query_sequence  query_length  prob_amp  true_label  prediction_result')
print('  [physicochemical] hydrophobicity_pct  net_charge  pos_charge_pct  neg_charge_pct')
for n in range(1, TOP_K + 1):
    print(f'  [hit {n}]           hit{n}_starpep_id  hit{n}_cosine_sim  hit{n}_sequence  hit{n}_length')
    print(f'                    hit{n}_activities  hit{n}_targets  hit{n}_source  hit{n}_modifications')
print('  [explanation]     biological_explanation')
print()
display(results_df[['query_sequence','prob_amp','prediction_result',
                     'hit1_starpep_id','hit1_cosine_sim','hit1_activities']].head(3))

Results saved -> /Users/deepugeorgejacob/Projects/amp-rag/results/amp_rag_results.csv
   Rows    : 358
   Columns : 39
   Size    : 1372.6 KB

Column layout:
  [query]           seq_idx  query_sequence  query_length  prob_amp  true_label  prediction_result
  [physicochemical] hydrophobicity_pct  net_charge  pos_charge_pct  neg_charge_pct
  [hit 1]           hit1_starpep_id  hit1_cosine_sim  hit1_sequence  hit1_length
                    hit1_activities  hit1_targets  hit1_source  hit1_modifications
  [hit 2]           hit2_starpep_id  hit2_cosine_sim  hit2_sequence  hit2_length
                    hit2_activities  hit2_targets  hit2_source  hit2_modifications
  [hit 3]           hit3_starpep_id  hit3_cosine_sim  hit3_sequence  hit3_length
                    hit3_activities  hit3_targets  hit3_source  hit3_modifications
  [explanation]     biological_explanation



,query_sequence,prob_amp,prediction_result,hit1_starpep_id,hit1_cosine_sim,hit1_activities
0,GIVDFAKKVVGGIRNALGI,0.799875,TP,starPep_00309,1.0,"Anti-Gram negative, Anti-Gram positive, Antimi..."
1,FISAIASMLGKFL,0.799397,TP,starPep_00267,1.0,"Anti-Gram positive, Antimicrobial, Antibacteri..."
2,FLPLVGKILSGLI,0.798918,TP,starPep_01728,1.0,"Antibacterial, Antimicrobial, Antifungal, Anti..."


In [ ]:
row = results_df.iloc[0]
print('=' * 80)
print(f"  seq_idx            : {row['seq_idx']}")
print(f"  query_sequence     : {row['query_sequence']}")
print(f"  prob_amp           : {row['prob_amp']:.4f}")
print(f"  prediction_result  : {row['prediction_result']}")
print(f"  net_charge         : {row['net_charge']:+d}")
print(f"  hydrophobicity_pct : {row['hydrophobicity_pct']}%")
print()
for n in range(1, TOP_K + 1):
    p = f'hit{n}_'
    print(f"  Hit #{n}: {row[p+'starpep_id']}  cosine_sim={row[p+'cosine_sim']:.2f}")
    print(f"    Sequence   : {row[p+'sequence']}")
    print(f"    Activities : {str(row[p+'activities'])[:80]}")
    print(f"    Targets    : {str(row[p+'targets'])[:80]}")
    print()
print('-' * 80)
print('  BIOLOGICAL EXPLANATION (Mistral-7B-Instruct, T=0):')
print('-' * 80)
print(row['biological_explanation'])
print('=' * 80)

  seq_idx            : 1
  query_sequence     : GIVDFAKKVVGGIRNALGI
  prob_amp           : 0.7999
  prediction_result  : TP
  net_charge         : +2
  hydrophobicity_pct : 52.6%

  Hit #1: starPep_00309  cosine_sim=1.00
    Sequence   : GIVDFAKKVVGGIRNALGI
    Activities : Anti-Gram negative, Anti-Gram positive, Antimicrobial, Antibacterial
    Targets    : Streptococcus uberis, Leuconostoc mesenteroides, Micrococcus luteus

  Hit #2: starPep_00302  cosine_sim=0.99
    Sequence   : GILDFAKTVVGGIRNALGI
    Activities : Antimicrobial, Anti-Gram positive, Anti-Gram negative, Antibacterial
    Targets    : Leuconostoc mesenteroides, Streptococcus uberis, Staphylococcus aureus, Listeria

  Hit #3: starPep_00295  cosine_sim=0.98
    Sequence   : GFVDLAKKVVGGIRNALGI
    Activities : Antibacterial, Antimicrobial, Anti-Gram negative, Anti-Gram positive
    Targets    : Streptococcus uberis, Leuconostoc mesenteroides, Staphylococcus aureus, Listeria

--------------------------------------------